An example that LLM sends back tool_use with multiple tool calls

disable_parallel_tool_use does the trick

```
response = client.messages.create(
    model="claude-opus-5-5",
    max_tokens=1024,
    tools=tools,
    # Ask for at most one tool call per turn.
    tool_choice={"type": "auto", "disable_parallel_tool_use": False},
    messages=messages,
)
```

In [18]:
from util.claude_client import get_claude_client_and_default_model

client = get_claude_client_and_default_model('deepseek')





llm profile name : deepseek , model : deepseek-flash


In [21]:
tools = [
    {
        'name': 'get_time',
        'description' : 'find the current time in a given timezone',
        'input_schema': {
            'type': 'object',
            'properties': {
                'timezone': {
                    'type': 'string',
                    'description' : 'time zone abbreviations, e.g. EST' ,
                    'default': 'UTC'},
            },
            'required': ['timezone']

        }

    },

    {
        'name': 'get_weather',
        'description': 'Get the current weather in a given location',
        'input_schema': {
            'type': 'object',
            'properties': {
                'location': {
                    'type': 'string',
                    'description' : 'The city and state, e.g. San Francisco, CA',
                }
            }
        },
        'required': ['location']
    }

]
messages = [
    {
        "role": "user",
        "content": "What's the weather in SF and NYC, and what time is it there?",
    }
]


def call_tool( tool_name : str , input:dict ):
    print( f'call tool : tool name: {tool_name}  , input: {input}')
    if tool_name == 'get_time':
        if "PDT" in str(input['timezone']):
            result = "2:30 PM PDT"
        elif "PST" in str(input['timezone']):
            result = "1:30 PM PDT"
        elif "EDT" in str(input['timezone']):
            result = "5:30 PM EDT"
        else:
            result = "4:30 PM EST"
    else:
        if "San Francisco" in str(input['location']):
            result = "San Francisco: 68°F, partly cloudy"
        else:
            result = "New York: 45°F, clear skies"
    return result


In [26]:
client  , llm= get_claude_client_and_default_model('deepseek')
while True:

    print("messages ######################### \n")
    print(messages)
    message = client.messages.create(
        max_tokens=1024,
        model=llm,
        messages = messages,
        tools=tools,
        tool_choice={ "type":"auto" , "disable_parallel_tool_use": False},
    )
    print( message.model_dump_json(indent=4) )
    if message.stop_reason == 'end_turn':
        break


    messages += [

        {
            "role": "assistant",
            "content": message.content
        }

    ]




    tool_uses = [block for block in message.content if block.type == 'tool_use']
    print(f"number of tools used: {len(tool_uses)}")
    for tool_use in tool_uses:
        tool_execution_result = call_tool(tool_use.name, tool_use.input)

        messages += [

            {
                'role': 'user',
                'content': [
                    {
                        'type':'tool_result',
                        'tool_use_id': tool_use.id,
                        'content': tool_execution_result
                    }
                ]
            }

        ]


print( message.model_dump_json(indent=4) )


llm profile name : deepseek , model : deepseek-flash
messages ######################### 

[{'role': 'user', 'content': "What's the weather in SF and NYC, and what time is it there?"}, {'role': 'assistant', 'content': [ThinkingBlock(signature='e5ce0738-1192-4d3d-bac5-1f0fd6ed650e', thinking='Need to get weather for SF and NYC, and time. SF is Pacific time, NYC is Eastern time.\n\nLet me make independent calls in one block.', type='thinking'), TextBlock(citations=None, text="I'll check the weather and current time for both cities.", type='text'), ToolUseBlock(id='call_00_7hk1UCwcCFGqll0rGz2g2199', caller=None, input={'location': 'San Francisco, CA'}, name='get_weather', type='tool_use', toolset_name=None), ToolUseBlock(id='call_01_4FI5qMO5ASqPSVxvasPp8312', caller=None, input={'location': 'New York, NY'}, name='get_weather', type='tool_use', toolset_name=None), ToolUseBlock(id='call_02_8Qsph4LTIAAF36ueZgvs8054', caller=None, input={'timezone': 'PST'}, name='get_time', type='tool_use', to

UnprocessableEntityError: Error code: 422 - {'error': {'message': 'Failed to deserialize the JSON body into the target type: messages[2].content: expected a string or a list at line 1 column 1047 (request_id: d0559b3f-fd66-478f-9f5c-5c7666363501)', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_request_error'}}